In [16]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml.paths import EXPERIMENT_RESULTS_FILE, EXPERIMENT_CONFIGS_FILE, EXPERIMENT_FEATURE_EFFECT, SELECTION_RESULTS_FILE, SELECTION_CONFIGS_FILE, SELECTION_FEATURE_EFFECT, TRAIN_PATH, TEST_PATH
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.configurations.exp_config import ALL_EXPERIMENTS
from titanic_ml.common.experiments.configurations.configuration import ALL_FE_CONFIG, ALL_FS_CONFIGS, LOGREG_FS_CONFIGS
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs, save_feature_effects, load_feature_effects
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard, model_progression, analyze_feature_effect, domain_best_by_model_with_baseline_delta, domain_best_by_model, recommended_by_domain_for_model
from titanic_ml.common.models.registry import MODEL_REGISTRY
from titanic_ml.feature_engineering import add_family_features, add_has_cabin, add_title, add_full_title_feature
from titanic_ml.common.data.eda import sample_dataframe

# import inspect
# print(ALL_FS_CONFIGS.__file__)
# print(inspect.getsource(ALL_FS_CONFIGS.overview))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
# print(ALL_FS_CONFIGS)
# print()
# print(ALL_EXPERIMENTS)

#### Feature selection pipeline and related functions:

In [18]:
TARGET = "Survived"
results_path = SELECTION_RESULTS_FILE
configs_path = SELECTION_CONFIGS_FILE
feature_effect_path = SELECTION_FEATURE_EFFECT

for experiment_name, exp_config in ALL_FS_CONFIGS.items():
    print(f"Experiment: {experiment_name}")



Experiment: fs__baseline__logreg
Experiment: fs01__age_cb03__logreg
Experiment: fs01__age_cb02__logreg
Experiment: fs02__family_cb07__logreg
Experiment: fs02__family_fe01__logreg
Experiment: fs03__cabin_fe04__logreg
Experiment: fs04__ticket_fe09_batch__logreg
Experiment: fs04__ticket_fe09_fitted__logreg
Experiment: fs04__ticket_fe09_full_context__logreg
Experiment: fs05__fare_fe08__logreg
Experiment: fs05__fare_cb05_fitted__logreg
Experiment: fs05__fare_cb05_full_context__logreg
Experiment: fs06__sex_pclass_fe12__logreg
Experiment: fs06__sex_pclass_cb08__logreg
Experiment: pr01__removing_family__logreg
Experiment: pr02__removing_ticket__logreg
Experiment: pr03__removing_non_raw_fare__logreg
Experiment: pr03__removing_fare__logreg
Experiment: pr03__removing_raw_fare__logreg
Experiment: pr04__removing_age__logreg
Experiment: pr05__removing_cabin__logreg
Experiment: pr06__removing_title__logreg
Experiment: prsc__age_cb02__logreg
Experiment: pr07__removing_sibSp_parch__logreg
Experiment: p

In [19]:
# # Debugging
# for key, values in ALL_FS_CONFIGS["fs__baseline__logreg"]['fs__baseline'].items():
#     print(f"Key: {key}")
#     print(f"Values : {values}")

In [20]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

exp_configs = ALL_FS_CONFIGS["pr05__removing_fe02__decision_tree"]
reference_group = 'pr03__removing_fe09_cb06_using_fe02'
print(exp_configs)
# # Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# # Uncomment to run all experiments and update results.

# for Name, exp_config in ALL_FS_CONFIGS.items():
#     print(f"Running {Name} experiments...")
#     print(f"Experiment config: {exp_config}")
#     group_name = Name.rsplit("__", 1)[0]
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True, context_df=test_df)
#     result_df = save_results(exp_result, path=results_path)
#     save_configs(exp_config, path=configs_path)
#     if group_name != 'baseline__raw':
#         comparison = compare_experiment_groups(
#             results_df=result_df,
#             reference_group="fs__baseline",
#             compare_groups=[group_name],
#         )
#         feature_effect = analyze_feature_effect(comparison)
#         save_feature_effects(feature_effect, path=feature_effect_path)

{'pr05__removing_fe02': {'pre_cv_feature_pipeline': [], 'pre_cv_scope': None, 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone'], 'onehot_features': ['Sex', 'Embarked', 'Title'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'feature_pipeline': [('title', TitleTransformer()), ('family_size', SumColumnsTransformer(constant=1, output_col='FamilySize',
                      source_cols=['SibSp', 'Parch'])), ('is_alone', IsOneTransformer(output_col='IsAlone', source_col='FamilySize'))], 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'stage': 'pr05', 'feature_group': 'removing_fe02', 'group': 'pr05__removing_fe02', 'domain': 'decision_tree', 'tags': set(), 'model_name': 'decision_tree', 'model_params': {'max_depth': 4, 'min_samples_leaf': 5, 'random_state': 42}, 'not

In [21]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    reference_group=reference_group,
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
    save=True,
    context_df=test_df,
    results_path=results_path,
    configs_path=configs_path,
    feature_effect_path=feature_effect_path,
    debug=True
)
# print("Workflow completed. Here are the results:")
# print("Comparison between baseline and feature engineering group:")
# print(workflow["comparison"])
# print("Summary of comparison:")
# print(workflow["summary"])
# print("Leaderboard:")
# print(workflow["leaderboard"])

Running config: {'pre_cv_feature_pipeline': [], 'pre_cv_scope': None, 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone'], 'onehot_features': ['Sex', 'Embarked', 'Title'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'feature_pipeline': [('title', TitleTransformer()), ('family_size', SumColumnsTransformer(constant=1, output_col='FamilySize',
                      source_cols=['SibSp', 'Parch'])), ('is_alone', IsOneTransformer(output_col='IsAlone', source_col='FamilySize'))], 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'stage': 'pr05', 'feature_group': 'removing_fe02', 'group': 'pr05__removing_fe02', 'domain': 'decision_tree', 'tags': set(), 'model_name': 'decision_tree', 'model_params': {'max_depth': 4, 'min_samples_leaf': 5, 'random_state': 42}, 'notes': 'Ex

In [22]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow, top_n=20)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()
# For combos:
all_results = load_results(path=results_path)
references = [reference_group]
for reference in references:
    comparison = compare_experiment_groups(
                results_df=all_results,
                reference_group=reference,
                compare_groups=[exp_configs],
            )
    print(f"Comparison summary:")
    print(comparison[["reference_group", "compare_group", "model_name", "test_accuracy_mean_delta", "test_f1_mean_delta"]].to_markdown())
    print()


Full workflow report:

Report
#### pr05__removing_fe02

_Description pending._

<details>
<summary>Conclusion</summary>


##### Interpretation

- Verdict: general_negative
- Discard
- Max delta: -0.003


##### Conclusion

_Conclusion pending._

</details>

<details>
<summary>Experiment details</summary>

##### Comparison vs pr03__removing_fe09_cb06_using_fe02

| reference_group                     | compare_group       | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------------------------|:--------------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| pr03__removing_fe09_cb06_using_fe02 | pr05__removing_fe02 | decision_tree |                          0.827 |                        0.824 |    

In [23]:
# raise to quickly run small experiments and reports
raise

RuntimeError: No active exception to reraise

In [ ]:
# Leaderboard without ablations
current_leaderboard = titanic_notes_leaderboard(all_results, top_n=11, selection="exclude")
print("Current leaderboard:")
print(current_leaderboard)

In [ ]:
# for model in MODEL_REGISTRY:
#     model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
#     print(f"Model progression for {model}:")
#     print(model_progression_df)
#     print()

#### Feature experiments pipeline and related functions:

In [ ]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_FE_CONFIG

results_path = EXPERIMENT_RESULTS_FILE
configs_path = EXPERIMENT_CONFIGS_FILE
feature_effect_path = EXPERIMENT_FEATURE_EFFECT


for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

print(ALL_EXPERIMENTS)


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

exp_configs = ALL_EXPERIMENTS["baseline__raw"]

# # Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# # Uncomment to run all experiments and update results.

# for Name, exp_config in ALL_EXPERIMENTS.items():
#     print(f"Running {Name} experiments...")
#     print(f"Experiment config: {exp_config}")
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True, context_df=test_df)
#     result_df = save_results(exp_result, path=results_path)
#     save_configs(exp_config, path=configs_path)
#     if Name != 'baseline__raw':
#         comparison = compare_experiment_groups(
#             results_df=result_df,
#             reference_group="baseline__raw",
#             compare_groups=[Name],
#         )
#         feature_effect = analyze_feature_effect(comparison)
#         save_feature_effects(feature_effect, path=feature_effect_path)

print(exp_configs)

In [ ]:
# print("Experiment Configurations:")
# print(exp_configs)
# for exp_config in exp_configs:
#     print(exp_config)

In [ ]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
    save=True,
    context_df=test_df,
    results_path=results_path,
    configs_path=configs_path,
    feature_effect_path=feature_effect_path
)
# print("Workflow completed. Here are the results:")
# print("Comparison between baseline and feature engineering group:")
# print(workflow["comparison"])
# print("Summary of comparison:")
# print(workflow["summary"])
# print("Leaderboard:")
# print(workflow["leaderboard"])

In [ ]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow, top_n=20)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()
# For combos:
all_results = load_results(path=results_path)
references = ['baseline__raw']
for reference in references:
    comparison = compare_experiment_groups(
                results_df=all_results,
                reference_group=reference,
                compare_groups=[exp_configs],
            )
    print(f"Comparison summary:")
    print(comparison[["reference_group", "compare_group", "model_name", "test_accuracy_mean_delta", "test_f1_mean_delta"]].to_markdown())
    print()


In [ ]:
# raise to quickly run small experiments and reports
raise

In [ ]:
# Leaderboard without ablations
current_leaderboard = titanic_notes_leaderboard(all_results, top_n=100, selection="exclude")
print("Current leaderboard:")
print(current_leaderboard)

In [ ]:
from titanic_ml.common.experiments.inspection import get_feature_pipeline, get_cv_fold, inspect_experiment_fold
from titanic_ml.common.experiments.runner import build_model_pipeline
for exp_name, exp_config in exp_configs.items():
    exp = exp_config
    break

model_pipeline = build_model_pipeline(exp)
for name, step in model_pipeline.steps:
    print(name, type(step).__name__)

In [ ]:
inspection = inspect_experiment_fold(
    df=train_df,
    experiments=ALL_EXPERIMENTS["test_family_ticket_full_context_config"],
    target="Survived",
    context_df=test_df,
    fold=0,
)

In [ ]:
inspection[
    "X_validation_engineered"
].columns

In [ ]:
train = inspection["X_train_cv_input"]
validation = inspection[
    "X_validation_engineered"
]

row = validation.iloc[0]

ticket = row["Ticket"]

print(
    "Ticket:",
    ticket,
)

print(
    "Training count:",
    train["Ticket"].eq(ticket).sum(),
)

print(
    "Engineered count:",
    row["TicketGroupSize"],
)

In [ ]:
validation_raw = inspection[
    "X_train_cv_input"
]

validation = inspection[
    "X_validation_engineered"
]

row = validation.iloc[0]
ticket = row["Ticket"]

print(
    validation_raw["Ticket"]
    .eq(ticket)
    .sum()
)

print(
    row["TicketGroupSize"]
)

In [ ]:
inspection["X_train_cv_input"]
inspection["X_validation_cv_input"]

In [ ]:
df = inspection[
    "X_validation_engineered"
]

(
    df["FamilySize"]
    ==
    df["SibSp"] + df["Parch"] + 1
).all()

In [ ]:
def compare_group_representations(
    df,
    family_col="FamilySize",
    ticket_col="TicketGroupSize",
    family_fare_col="Fare/FamilySize",
    ticket_fare_col="Fare/TicketMember",
):
    same_group_size = df[family_col].eq(df[ticket_col])

    return {
        "group_size_equal_rate": same_group_size.mean(),

        "group_size_mae": (
            df[family_col] - df[ticket_col]
        ).abs().mean(),

        "fare_pearson": df[
            [family_fare_col, ticket_fare_col]
        ].corr(method="pearson").iloc[0, 1],

        "fare_spearman": df[
            [family_fare_col, ticket_fare_col]
        ].corr(method="spearman").iloc[0, 1],

    "family_unique": df[family_fare_col].nunique(),

    "ticket_unique": df[ticket_fare_col].nunique(),
    
    "exact_fare_equal_rate": (
        df[family_fare_col] == df[ticket_fare_col]
    ).mean(),
}


In [ ]:
print("Comparing family and ticket group representations:")
compare_group = compare_group_representations(inspection["X_validation_engineered"])
for key, value in compare_group.items():
    print(f"{key}: {value}")

In [ ]:
raise

In [ ]:
thresholds = {
    "accuracy": 0.003,
    "f1": -0.01,
}

for model in MODEL_REGISTRY:

    result = recommended_by_domain_for_model(
        results_df=all_results,
        model_name=model,
        thresholds=thresholds,
    )

    print(f"\nBest candidates for {model}")
    print("=" * 50)

    print("\nRecommended:")
    for recommendation in result["recommended"]:
        print(
            recommendation["domain"],
            "->",
            recommendation["group"],
            recommendation["deltas"],
        )
    print()
    print("recommended list:")
    for recommendation in result["recommended"]:
        print(recommendation["group"], end=", ")
    print()
    print("\nFull domain results:")
    print(result["df"].to_markdown(index=False))

In [ ]:
# import pprint
# Feature_effect = analyze_feature_effect(workflow['comparison'])
# pprint.pprint(Feature_effect)

In [ ]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
# exp_result = run_experiments(train_df, exp_config, target=TARGET,)
# exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

In [ ]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [ ]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

In [ ]:
# print(workflow["all_results"])

In [ ]:
# for model in MODEL_REGISTRY:
#     model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
#     print(f"Model progression for {model}:")
#     print(model_progression_df)
#     print()

Old Eda practice

In [ ]:
# eda = run_eda(train_df, target=TARGET, display=True, head=3,random=4, tail=3)
# print(eda)

In [ ]:
# print(eda)

In [ ]:
# print(eda['dataFrame_health'].to_markdown())

In [ ]:
# print(eda["dataFrame_summary"].to_markdown())

In [ ]:
# for col in eda['categorical_summary']:
#     sample = sample_dataframe(eda['categorical_summary'][col], head=1, random=3, tail=1)
#     sample_df = pd.concat(
#         [sample[k] for k in ['head', 'random', 'tail']],
#         ignore_index=False
#     )
#     print(sample_df.to_markdown())
#     print()

In [ ]:
# for col in eda['numerical_summary']:
#     print(f"Numerical column: {col}")
#     print(eda['numerical_summary'][col].to_markdown())
#     print()

In [ ]:
# print("Correlation matrix:")
# print(eda["correlation_matrix"].to_markdown())

In [ ]:
# print('Correlation with the target variable:')
# print(eda["target_correlation"].to_markdown())

In [ ]:
# for col in eda["categorical_rare"]:
#     print(f"Categorical column with rare values: {col}")
#     print(eda["categorical_rare"][col])
#     print()

In [ ]:
# # print(eda['sample'])
# sample_df = pd.concat(
#     [eda['sample'][k] for k in ['head', 'random', 'tail']],
#     ignore_index=True
# )
# print(sample_df.to_markdown(index=False))

In [ ]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(paths.TRAIN_PATH)
test_df = pd.read_csv(paths.TEST_PATH)

exp_configs = ALL_EXPERIMENTS["baseline__raw"]

In [29]:
from titanic_ml.common.experiments.runner import apply_pre_cv_feature_pipeline, build_model_pipeline

In [30]:
from sklearn.model_selection import GridSearchCV
import pandas as pd


def _run_single_hpo(
    df,
    exp,
    target,
    context_df=None,
):
    """
    Run a grid search using an existing experiment configuration.

    Returns
    -------
    search : GridSearchCV
        Fitted search object containing the best estimator,
        best parameters, and complete CV results.

    results : pd.DataFrame
        One row per evaluated hyperparameter combination.
    """

    # -------------------------------------------------
    # 1. Validate configuration
    # -------------------------------------------------

    search_config = exp.get("hyperparameter_search")

    if not isinstance(search_config, dict):
        raise ValueError(
            "Experiment must contain a "
            "'hyperparameter_search' dictionary."
        )

    if search_config.get("method") != "grid":
        raise ValueError(
            "Only 'grid' hyperparameter search "
            "is currently supported."
        )

    param_grid = search_config.get("param_grid")

    if not isinstance(param_grid, (dict, list)) or not param_grid:
        raise ValueError(
            "A non-empty 'param_grid' is required."
        )

    if target not in df.columns:
        raise ValueError(
            f"Target column '{target}' not found."
        )

    # -------------------------------------------------
    # 2. Prepare predictors and target
    # -------------------------------------------------

    X = df.drop(columns=[target])
    y = df[target]

    if exp.get("pre_cv_feature_pipeline"):
        X = apply_pre_cv_feature_pipeline(
            X=X,
            exp=exp,
            context_df=context_df,
        )

    # -------------------------------------------------
    # 3. Build model pipeline
    # -------------------------------------------------

    model_pipeline = build_model_pipeline(exp)

    # Validate searched parameter names
    valid_params = model_pipeline.get_params(deep=True)

    grids = param_grid if isinstance(param_grid, list) else [param_grid]

    for grid in grids:
        if not isinstance(grid, dict):
            raise TypeError(
                "Each parameter grid must be a dictionary."
            )

        for parameter in grid:
            if parameter not in valid_params:
                raise ValueError(
                    f"Unknown pipeline parameter: '{parameter}'."
                )

    # -------------------------------------------------
    # 4. Read evaluation configuration
    # -------------------------------------------------

    evaluation = exp["evaluation"]

    if evaluation.get("method", "cross_validation") != "cross_validation":
        raise ValueError(
            "Hyperparameter search requires cross-validation."
        )

    scoring = evaluation.get("scoring", ["accuracy"])

    if isinstance(scoring, str):
        scoring = [scoring]

    if not scoring:
        raise ValueError("At least one scoring metric is required.")

    refit = search_config.get("refit", "accuracy")

    if refit not in scoring:
        raise ValueError(
            f"Refit metric '{refit}' is not included in scoring."
        )

    # -------------------------------------------------
    # 5. Execute search
    # -------------------------------------------------

    search = GridSearchCV(
        estimator=model_pipeline,
        param_grid=param_grid,
        scoring=scoring,
        refit=refit,
        cv=evaluation.get("cv", 5),
        n_jobs=evaluation.get("n_jobs", -1),
        return_train_score=evaluation.get(
            "return_train_score", True
        ),
        error_score="raise",
    )

    search.fit(X, y)

    # -------------------------------------------------
    # 6. Collect results
    # -------------------------------------------------

    results = pd.DataFrame(search.cv_results_)

    results.insert(0, "experiment", exp["name"])

    return search, results

In [ ]:
from titanic_ml.common.experiments.utils import get_experiment_configs

def run_hyperparameter_search(
    df,
    experiments,
    target,
    context_df=None,
):
    """
    Run hyperparameter searches for a group of experiments.

    Returns
    -------
    searches : dict
        Fitted search objects keyed by experiment name.

    results : pd.DataFrame
        Combined candidate-level CV results.
    """

    if not isinstance(experiments, dict):
        raise TypeError(
            "experiments must be a dictionary of experiment configurations."
        )

    if "model_name" in experiments:
        raise ValueError(
            "Received a single experiment configuration. "
            "Wrap it with config_to_group() before running HPO."
        )

    experiment_iter = get_experiment_configs(experiments)

    searches = {}
    result_frames = []

    for exp in experiment_iter:
        search, results = _run_single_hpo(
            df=df,
            exp=exp,
            target=target,
            context_df=context_df,
        )

        experiment_name = exp["name"]

        if experiment_name in searches:
            raise ValueError(
                f"Duplicate experiment name: '{experiment_name}'."
            )

        searches[experiment_name] = search
        result_frames.append(results)

    combined_results = pd.concat(
        result_frames,
        ignore_index=True,
    ) if result_frames else pd.DataFrame()

    return searches, combined_results

In [32]:
import copy
from titanic_ml.common.experiments.config_creation import config_to_group


best_logreg_fs = ALL_FS_CONFIGS["pr03__removing_fare__logreg"]

logreg_selected_features = copy.deepcopy(
    next(iter(best_logreg_fs.values()))
)

logreg_hpo01 = copy.deepcopy(logreg_selected_features)

logreg_hpo01.update({
    "name": "hpo01__regularization__logreg",
    "stage": "hpo01",
    "feature_group": "regularization",
    "group": "hpo01__regularization",
    "source_experiment": logreg_selected_features["name"],
    "hyperparameter_search": {
        "method": "grid",
        "param_grid": {
            "model__C": [0.01, 0.1, 1, 10, 100],
        },
        "refit": "accuracy",
    },
})

hpo01_configs = config_to_group(logreg_hpo01)

In [33]:
searches, results = run_hyperparameter_search(
    df=train_df,
    experiments=hpo01_configs,
    target="Survived",
)

search = next(iter(searches.values()))

print(search.best_params_)
print(search.best_score_)

{'model__C': 1}
0.840631473228297


In [35]:
print("Best parameters:", search.best_params_)
print("Best CV accuracy:", search.best_score_)

display(
    results[
        [
            "param_model__C",
            "mean_test_accuracy",
            "std_test_accuracy",
            "mean_test_f1",
            "std_test_f1",
            "mean_train_accuracy",
            "rank_test_accuracy",
        ]
    ].sort_values("rank_test_accuracy")
)

Best parameters: {'model__C': 1}
Best CV accuracy: 0.840631473228297


,param_model__C,mean_test_accuracy,std_test_accuracy,mean_test_f1,std_test_f1,mean_train_accuracy,rank_test_accuracy
2,1.00,0.840631,0.015696,0.786753,0.029488,0.845118,1
3,10.00,0.832773,0.017543,0.777235,0.032235,0.844276,2
4,100.00,0.831655,0.017708,0.775488,0.031964,0.843434,3
1,0.10,0.827136,0.014642,0.765646,0.027621,0.832492,4
0,0.01,0.808079,0.020241,0.717495,0.027926,0.822672,5


In [42]:
logreg_hpo02 = copy.deepcopy(logreg_hpo01)

logreg_hpo02.update({
    "name": "hpo02__refine_c__logreg",
    "stage": "hpo02",
    "feature_group": "refine_c",
    "group": "hpo02__refine_c",
})

logreg_hpo02["hyperparameter_search"]["param_grid"] = {
    "model__C": [
        0.2, 0.4, 0.6, 0.8, 1.0, 1.2,
        1.5, 2.0, 3.0, 5.0,
    ],
}

hpo02_configs = config_to_group(logreg_hpo02)

searches, results = run_hyperparameter_search(
    df=train_df,
    experiments=hpo02_configs,
    target="Survived",
)

search = next(iter(searches.values()))

print("Best parameters:", search.best_params_)
print("Best CV accuracy:", search.best_score_)

Best parameters: {'model__C': 1.0}
Best CV accuracy: 0.840631473228297


In [43]:
def summarize_hpo_results(results):
    columns = {
        "experiment": "Experiment",
        "param_model__C": "C",
        "mean_test_accuracy": "CV Accuracy",
        "std_test_accuracy": "Accuracy Std",
        "mean_test_f1": "CV F1",
        "std_test_f1": "F1 Std",
        "mean_train_accuracy": "Train Accuracy",
        "rank_test_accuracy": "Rank",
    }

    available = {
        key: value
        for key, value in columns.items()
        if key in results.columns
    }

    summary = results[list(available)].rename(
        columns=available
    )

    return summary.sort_values(
        ["CV Accuracy", "CV F1"],
        ascending=False,
    ).round(5)

In [44]:
summarize_hpo_results(results)

,Experiment,C,CV Accuracy,Accuracy Std,CV F1,F1 Std,Train Accuracy,Rank
4,hpo02__refine_c__logreg,1.0,0.84063,0.01570,0.78675,0.02949,0.84512,1
2,hpo02__refine_c__logreg,0.6,0.84063,0.01361,0.78624,0.02709,0.84596,2
3,hpo02__refine_c__logreg,0.8,0.84063,0.01361,0.78624,0.02709,0.84456,2
1,hpo02__refine_c__logreg,0.4,0.83726,0.01179,0.78165,0.02373,0.84371,4
6,hpo02__refine_c__logreg,1.5,0.83614,0.01960,0.78117,0.03539,0.84484,5
7,hpo02__refine_c__logreg,2.0,0.83614,0.01960,0.78117,0.03539,0.84428,5
5,hpo02__refine_c__logreg,1.2,0.83502,0.01963,0.77942,0.03506,0.84484,7
9,hpo02__refine_c__logreg,5.0,0.83277,0.01754,0.77724,0.03223,0.84540,8
8,hpo02__refine_c__logreg,3.0,0.83165,0.01777,0.77546,0.03213,0.84456,9
0,hpo02__refine_c__logreg,0.2,0.82939,0.01548,0.77003,0.02797,0.83810,10


In [45]:
def summarize_hpo_results(results, decimals=5):
    parameter_columns = [
        column
        for column in results.columns
        if column.startswith("param_")
    ]

    metric_columns = [
        "mean_test_accuracy",
        "std_test_accuracy",
        "mean_test_f1",
        "std_test_f1",
        "mean_train_accuracy",
        "rank_test_accuracy",
    ]

    selected_columns = [
        "experiment",
        *parameter_columns,
        *[
            column
            for column in metric_columns
            if column in results.columns
        ],
    ]

    summary = results[selected_columns].copy()

    summary = summary.rename(
        columns={
            "experiment": "Experiment",
            **{
                column: column.removeprefix("param_model__")
                for column in parameter_columns
            },
            "mean_test_accuracy": "CV Accuracy",
            "std_test_accuracy": "Accuracy Std",
            "mean_test_f1": "CV F1",
            "std_test_f1": "F1 Std",
            "mean_train_accuracy": "Train Accuracy",
            "rank_test_accuracy": "Rank",
        }
    )

    return (
        summary
        .sort_values(
            ["CV Accuracy", "CV F1"],
            ascending=False,
        )
        .round(decimals)
        .reset_index(drop=True)
    )

In [52]:
logreg_hpo03 = copy.deepcopy(logreg_hpo02)

logreg_hpo03.update({
    "name": "hpo03__regularization_type__logreg",
    "stage": "hpo03",
    "feature_group": "regularization_type",
    "group": "hpo03__regularization_type",
})

logreg_hpo03["hyperparameter_search"]["param_grid"] = [
    {
        "model__solver": ["saga"],
        "model__penalty": ["l1", "l2"],
        "model__C": [0.4, 0.6, 0.8, 1.0, 1.5],
    },
    {
        "model__solver": ["saga"],
        "model__penalty": ["elasticnet"],
        "model__l1_ratio": [0.25, 0.5, 0.75],
        "model__C": [0.4, 0.6, 0.8, 1.0, 1.5],
    },
]

hpo03_configs = config_to_group(logreg_hpo03)

searches, results = run_hyperparameter_search(
    df=train_df,
    experiments=hpo03_configs,
    target="Survived",
)

search = next(iter(searches.values()))

print("Best parameters:", search.best_params_)
print("Best CV accuracy:", search.best_score_)

display(summarize_hpo_results(results))

Best parameters: {'model__C': 0.8, 'model__penalty': 'l2', 'model__solver': 'saga'}
Best CV accuracy: 0.8417487916640513


,Experiment,C,penalty,solver,l1_ratio,CV Accuracy,Accuracy Std,CV F1,F1 Std,Train Accuracy,Rank
0,hpo03__regularization_type__logreg,0.8,l2,saga,NaN,0.84175,0.01567,0.78790,0.02976,0.84484,1
1,hpo03__regularization_type__logreg,0.8,elasticnet,saga,0.25,0.84063,0.01361,0.78672,0.02785,0.84456,2
2,hpo03__regularization_type__logreg,0.6,l2,saga,NaN,0.84063,0.01361,0.78624,0.02709,0.84596,2
3,hpo03__regularization_type__logreg,1.0,l2,saga,NaN,0.83951,0.01649,0.78558,0.03027,0.84540,4
4,hpo03__regularization_type__logreg,1.0,elasticnet,saga,0.25,0.83951,0.01649,0.78558,0.03027,0.84428,4
5,hpo03__regularization_type__logreg,0.8,elasticnet,saga,0.50,0.83950,0.01452,0.78613,0.02893,0.84456,6
6,hpo03__regularization_type__logreg,0.6,elasticnet,saga,0.25,0.83950,0.01274,0.78565,0.02636,0.84400,7
7,hpo03__regularization_type__logreg,1.0,elasticnet,saga,0.50,0.83726,0.01549,0.78294,0.03129,0.84428,8
8,hpo03__regularization_type__logreg,0.4,l2,saga,NaN,0.83726,0.01179,0.78165,0.02373,0.84400,8
9,hpo03__regularization_type__logreg,1.0,elasticnet,saga,0.75,0.83614,0.01560,0.78122,0.03085,0.84400,10
